# X-UMX 오케스트라 음원 분리 학습

4개 악기군 모델을 기존 STFT·weighted SDR 손실로 학습합니다.


In [ ]:
import os, sys, json, importlib.util
from pathlib import Path

# 네 파일을 같은 폴더에 둡니다. Colab에서는 /content를 사용합니다.
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
PROJECT_ROOT = Path(os.environ.get("ENSEMBLE_PROJECT_ROOT", "/content" if IN_COLAB else ".")).expanduser().resolve()

def ensure_competition_kit(project_root):
    import shutil
    import stat
    import tempfile
    import zipfile
    from pathlib import PurePosixPath

    kit = project_root / "competition-kit"
    required = (
        "ensemble_benchmark/local_data.py",
        "ensemble_benchmark/runtime.py",
        "ensemble_benchmark/baseline.py",
        "manifests/selection.json",
        "requirements.txt",
        "ensemble_benchmark/losses.py",
        "ensemble_benchmark/evaluation_metrics.py",
        "distribution.json",
    )
    if kit.is_dir():
        if not all((kit / name).is_file() for name in required):
            raise FileNotFoundError(
                f"코드 폴더가 오래되었거나 불완전합니다: {kit}. 최신 competition-kit을 업로드하세요."
            )
        return kit

    archive = project_root / "competition-kit.zip"
    if not archive.is_file():
        raise FileNotFoundError(
            f"최신 competition-kit 폴더 또는 {archive}를 준비하세요."
        )
    with zipfile.ZipFile(archive) as zf:
        entries = []
        names = set()
        for item in zf.infolist():
            path = PurePosixPath(item.filename)

            if (
                path.is_absolute()
                or ".." in path.parts
                or "\\" in item.filename
                or ":" in item.filename
                or stat.S_ISLNK(item.external_attr >> 16)
            ):
                raise ValueError(f"허용하지 않는 ZIP 경로: {item.filename}")

            if "__MACOSX" in path.parts or path.name == ".DS_Store" or item.is_dir():
                continue

            name = str(path)
            if name in names:
                raise ValueError(f"중복 ZIP 경로: {name}")
            names.add(name)
            entries.append((item, name))

        prefixes = [
            prefix
            for prefix in ("", "competition-kit/")
            if all(prefix + name in names for name in required)
        ]

        if len(prefixes) != 1:
            raise ValueError(
                "최신 코드가 들어 있는 competition-kit.zip인지 확인하세요."
            )

        prefix = prefixes[0]
        print("코드 압축 해제:", archive, flush=True)
        with tempfile.TemporaryDirectory(
            prefix=".kit-extract-", dir=project_root
        ) as tmp:
            staging = Path(tmp) / "competition-kit"
            staging.mkdir()
            for item, name in entries:
                if not name.startswith(prefix):
                    continue
                dest = staging / name[len(prefix) :]
                dest.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(item) as src, dest.open("xb") as dst:
                    shutil.copyfileobj(src, dst)
            if kit.exists():
                raise FileExistsError(f"대상 폴더를 덮어쓰지 않습니다: {kit}")
            staging.rename(kit)
    return kit

KIT = ensure_competition_kit(PROJECT_ROOT)
for name, module in list(sys.modules.items()):
    if name == "ensemble_benchmark" or name.startswith("ensemble_benchmark."):
        origin = getattr(module, "__file__", None)
        if origin and not Path(origin).resolve().is_relative_to(KIT.resolve()):
            raise RuntimeError("다른 버전의 코드가 로드되어 있습니다. 커널을 재시작하세요.")
sys.path.insert(0, str(KIT))


In [ ]:
# 필요한 라이브러리가 없으면 설치합니다.
import subprocess
required = ("torch", "numpy", "scipy", "soundfile", "norbert", "museval", "tqdm")
if any(importlib.util.find_spec(name) is None for name in required):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(KIT / "requirements.txt")])

import torch
from ensemble_benchmark.local_data import resolve_prepared
from ensemble_benchmark.runtime import build_local_release

# 기존 데이터 폴더를 사용하거나 dataset.zip을 자동으로 풉니다.
DATA_ROOT = resolve_prepared(PROJECT_ROOT, explicit=os.environ.get("ENSEMBLE_PREPARED"), archive_root=PROJECT_ROOT)
SMOKE = False  # True: 작은 모델로 연결 확인
RUN_NAME = os.environ.get("ENSEMBLE_RUN", "local-smoke-check" if SMOKE else "local-baseline-b16")
RUNS_ROOT = Path(os.environ.get("ENSEMBLE_RUNS", str(PROJECT_ROOT / "runs"))).expanduser().resolve()
RUN_DIR = RUNS_ROOT / RUN_NAME
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
if DEVICE == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
print("데이터:", DATA_ROOT, "\n저장:", RUN_DIR, "\n장치:", DEVICE)
release = build_local_release(DATA_ROOT, smoke=SMOKE, excluded_train_ids=("sod_507", "sod_968"))
print("Train:", release["train_count"], "Validation:", release["validation_count"])


In [ ]:
from ensemble_benchmark.runtime import train_local

# 같은 run을 다시 실행하면 저장된 체크포인트에서 재개합니다.
EPOCHS = 1
BATCH_SIZE = 16
checkpoint = train_local(
    release, DATA_ROOT, RUN_DIR, device=DEVICE, epochs=EPOCHS,
    tiny=SMOKE, steps_per_shard=1 if SMOKE else None,
    batch_size=1 if SMOKE else BATCH_SIZE,
    samples_per_track=1 if SMOKE else 64,
    segment_seconds=0.25 if SMOKE else 6.0,
    save_every=1 if SMOKE else 50,
    loss_type="baseline",
)
print("모델:", checkpoint)


In [ ]:
# 선택: 실행 결과를 ZIP으로 저장합니다.
from ensemble_benchmark.export import export_run
print(export_run(RUN_DIR, include_predictions=False))
